In [0]:
from pyspark.sql import SparkSession

spark.sql("CREATE DATABASE IF NOT EXISTS flight_data_silver")
spark.sql("USE flight_data_silver")

DataFrame[]

In [0]:
%sql
CREATE TABLE IF NOT EXISTS flight_data_silver.dimairlines(
  airline_id string,
  airline_name string,
  airline_name_OLD string,
  airline_name_effective_date TIMESTAMP
);

MERGE INTO flight_data_silver.dimairlines AS dima
USING flight_data_bronze.airlines AS a
ON dima.airline_id = a.airline_id
WHEN MATCHED AND(
    dima.airline_name <> a.airline_name
) THEN
UPDATE SET
    dima.airline_name_OLD = (CASE WHEN dima.airline_name <> a.airline_name
                             THEN dima.airline_name
                             ELSE dima.airline_name_OLD END),
    dima.airline_name = a.airline_name,
    dima.airline_name_effective_date = (CASE WHEN dima.airline_name <> a.airline_name
                                        THEN current_timestamp()
                                        ELSE dima.airline_name_effective_date END)
WHEN NOT MATCHED
THEN INSERT(
    airline_id, airline_name, airline_name_OLD, airline_name_effective_date)
VALUES(
    a.airline_id, a.airline_name, NULL, current_timestamp())

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
14,0,0,14


In [0]:
%sql
SELECT * FROM flight_data_silver.dimairlines

airline_id,airline_name,airline_name_OLD,airline_name_effective_date
UA,United Air Lines Inc.,null,2025-12-12T11:26:19.590Z
AA,American Airlines Inc.,null,2025-12-12T11:26:19.590Z
US,US Airways Inc.,null,2025-12-12T11:26:19.590Z
F9,Frontier Airlines Inc.,null,2025-12-12T11:26:19.590Z
B6,JetBlue Airways,null,2025-12-12T11:26:19.590Z
OO,Skywest Airlines Inc.,null,2025-12-12T11:26:19.590Z
AS,Alaska Airlines Inc.,null,2025-12-12T11:26:19.590Z
NK,Spirit Air Lines,null,2025-12-12T11:26:19.590Z
WN,Southwest Airlines Co.,null,2025-12-12T11:26:19.590Z
DL,Delta Air Lines Inc.,null,2025-12-12T11:26:19.590Z
